In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

In [2]:
def evaluate_models(X, y, dataset_name):
    imputer = SimpleImputer(strategy="mean")
    X_imp = imputer.fit_transform(X)
    
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "SVM (RBF)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", probability=True, random_state=42))
        ]),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "KNN": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5))
        ]),
        "Random Forest (Ours)": RandomForestClassifier(
            n_estimators=200, random_state=42
        )
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []
    
    print(f"\n{'='*70}")
    print(f"  Dataset: {dataset_name}")
    print(f"{'='*70}")
    print(f"  {'Model':<28} {'Accuracy':>12} {'Std':>8} {'AUROC':>10} {'Std':>8}")
    print(f"  {'-'*66}")
    
    for name, model in models.items():
        acc  = cross_val_score(model, X_imp, y, cv=skf, scoring="accuracy")
        auc  = cross_val_score(model, X_imp, y, cv=skf, scoring="roc_auc")
        results.append({
            "Model": name,
            "Dataset": dataset_name,
            "Accuracy Mean": acc.mean(),
            "Accuracy Std":  acc.std(),
            "AUROC Mean":    auc.mean(),
            "AUROC Std":     auc.std()
        })
        marker = " <-- OURS" if "Random Forest" in name else ""
        print(f"  {name:<28} {acc.mean():>11.4f} {acc.std():>8.4f} "
              f"{auc.mean():>10.4f} {auc.std():>8.4f}{marker}")
    
    return pd.DataFrame(results)

In [3]:
pk_cols = ["pitch", "hnr", "jitter", "shimmer", "nhr", "rpde", "dfa"]

df_pk = pd.read_csv("../data/parkinsons_tabular/parkinsons.csv").rename(columns={
    "MDVP:Fo(Hz)": "pitch", "HNR": "hnr",
    "MDVP:Jitter(Abs)": "jitter", "MDVP:Shimmer": "shimmer",
    "NHR": "nhr", "RPDE": "rpde", "DFA": "dfa"
})
X_pk = df_pk[pk_cols]
y_pk = df_pk["status"]

results_pk = evaluate_models(X_pk, y_pk, "Parkinson's Disease")


  Dataset: Parkinson's Disease
  Model                            Accuracy      Std      AUROC      Std
  ------------------------------------------------------------------
  Logistic Regression               0.8308   0.0476     0.8255   0.0515
  SVM (RBF)                         0.8769   0.0497     0.8939   0.0630
  Decision Tree                     0.8359   0.0641     0.7920   0.0922
  KNN                               0.8923   0.0410     0.9668   0.0066
  Random Forest (Ours)              0.8872   0.0384     0.9494   0.0197 <-- OURS


In [4]:
feature_cols = [
    "pitch", "spectral_centroid", "zcr",
    "jitter", "shimmer", "hnr",
    "mfcc_1","mfcc_2","mfcc_3","mfcc_4","mfcc_5","mfcc_6","mfcc_7",
    "mfcc_8","mfcc_9","mfcc_10","mfcc_11","mfcc_12","mfcc_13"
]

df_resp = pd.read_csv("../data/voice_features/voice_dataset_labeled_full.csv").dropna()
X_resp = df_resp[feature_cols]
y_resp = df_resp["label"]

results_resp = evaluate_models(X_resp, y_resp, "Respiratory Abnormality")


  Dataset: Respiratory Abnormality
  Model                            Accuracy      Std      AUROC      Std
  ------------------------------------------------------------------
  Logistic Regression               0.6386   0.0045     0.6865   0.0143
  SVM (RBF)                         0.6934   0.0115     0.7534   0.0075
  Decision Tree                     0.5895   0.0225     0.5887   0.0227
  KNN                               0.6770   0.0196     0.7311   0.0188
  Random Forest (Ours)              0.6936   0.0138     0.7555   0.0124 <-- OURS


In [5]:
df_stress = pd.read_csv("../data/voice_features/stress_dataset.csv").dropna()
X_stress = df_stress[feature_cols]
y_stress = df_stress["label"]

results_stress = evaluate_models(X_stress, y_stress, "Psychological Stress")


  Dataset: Psychological Stress
  Model                            Accuracy      Std      AUROC      Std
  ------------------------------------------------------------------
  Logistic Regression               0.8021   0.0271     0.8594   0.0241
  SVM (RBF)                         0.8305   0.0198     0.9110   0.0213
  Decision Tree                     0.7756   0.0234     0.7738   0.0242
  KNN                               0.8494   0.0220     0.9141   0.0202
  Random Forest (Ours)              0.8362   0.0262     0.9184   0.0173 <-- OURS


In [6]:
df_depr = pd.read_csv("../data/voice_features/depression_dataset.csv").dropna()
X_depr = df_depr[feature_cols]
y_depr = df_depr["label"]

results_depr = evaluate_models(X_depr, y_depr, "Depression Indicators")


  Dataset: Depression Indicators
  Model                            Accuracy      Std      AUROC      Std
  ------------------------------------------------------------------
  Logistic Regression               0.7017   0.0321     0.7587   0.0302
  SVM (RBF)                         0.7803   0.0437     0.8493   0.0277
  Decision Tree                     0.6885   0.0224     0.6872   0.0253
  KNN                               0.7812   0.0231     0.8554   0.0242
  Random Forest (Ours)              0.7633   0.0295     0.8555   0.0320 <-- OURS


In [7]:
all_results = pd.concat([
    results_pk, results_resp, results_stress, results_depr
], ignore_index=True)

print("\n\nCOMPLETE BASELINE COMPARISON TABLE")
print("="*70)
pivot = all_results.pivot_table(
    index="Model",
    columns="Dataset",
    values=["Accuracy Mean", "AUROC Mean"]
).round(4)
print(pivot.to_string())

all_results.to_csv("../results/baseline_comparison.csv", index=False)
print("\nSaved: results/baseline_comparison.csv")



COMPLETE BASELINE COMPARISON TABLE
                                AUROC Mean                                                                          Accuracy Mean                                                                 
Dataset              Depression Indicators Parkinson's Disease Psychological Stress Respiratory Abnormality Depression Indicators Parkinson's Disease Psychological Stress Respiratory Abnormality
Model                                                                                                                                                                                             
Decision Tree                       0.6872              0.7920               0.7738                  0.5887                0.6885              0.8359               0.7756                  0.5895
KNN                                 0.8554              0.9668               0.9141                  0.7311                0.7812              0.8923               0.8494             